# Scheduled Climate Bias-Correction Pipeline (unattended)

This is a non-interactive twin of `run_interactive.ipynb`, built to run on
Colab Pro's **Scheduled notebooks** feature (Runtime menu -> Schedule /
the clock icon in the left sidebar of a notebook saved in Google Drive) --
i.e. it runs on a timer with nobody watching, so it can't use file-upload
widgets, a "Run" button, or the interactive `ee.Authenticate()` browser
login. Every parameter is a plain value in the CONFIG cell below instead
of a form.

## One-time setup (do this once, before scheduling)

1. **Put your shapefile on Google Drive** (not a local upload -- there's
   nobody here to upload it each run) and note its path, e.g.
   `/content/drive/MyDrive/aoi/AJK.shp`.

2. **Create a GEE-enabled service account** (this replaces the interactive
   Earth Engine login, which needs a live browser session):
   - Google Cloud Console -> IAM & Admin -> Service Accounts -> Create.
   - Grant it access to Earth Engine for your `gee_project_id` (register
     the service account at https://code.earthengine.google.com/register
     if your project needs it, or add it as an Earth Engine user in your
     GCP project).
   - Create a JSON key for it and download the file.

3. **Store the service account's credentials as Colab Secrets** (the
   padlock icon in the left sidebar) rather than pasting them into this
   notebook -- scheduled runs save a read-only copy of the notebook to
   Drive on every run, so anything hardcoded here would be duplicated
   every time:
   - Secret name `GEE_SA_EMAIL` -> the service account's email address.
   - Secret name `GEE_SA_KEY_JSON` -> the **entire contents** of the
     downloaded JSON key file, pasted as one value.
   - Toggle "Notebook access" on for both so this notebook can read them.

4. **Save this notebook to Google Drive**, open it at least once from
   Google Drive inside colab.research.google.com (Colab only allows
   scheduling notebooks you've edited within Colab itself), then use the
   **Schedule** button in the left sidebar to set the frequency.

Every run's output still goes to `output_dir` below (put that on Drive
too, so you can find it after an unattended run) -- see
`agreement_utils.py` / `template_excel_utils.py` for what gets written
there.

## 1. Setup: mount Drive, clone repo, install deps

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!git clone https://github.com/awan-geospatial1/climate-downscaling.git /content/climate-downscaling
!pip install -q -r /content/climate-downscaling/requirements.txt --upgrade

# Add to Python path
import sys
sys.path.insert(0, '/content/climate-downscaling')

## 2. Config -- edit these values, then leave the notebook alone

This replaces every widget from `run_interactive.ipynb` with a plain
value. There's no file-upload here on purpose -- `shapefile_path` must
already point at a file on Drive (step 1 above).

In [ ]:
params = {
    'shapefile_path': '/content/drive/MyDrive/aoi/AJK.shp',   # <-- your Drive shapefile path
    'buffer_km': 25.0,
    'gee_project_id': 'your-gcp-project-id',                  # <-- your GCP project ID

    'models': ['EC-Earth3', 'CNRM-CM6-1', 'GFDL-ESM4', 'MPI-ESM1-2-LR', 'GISS-E2-1-G'],
    'scenarios': ['ssp245', 'ssp585'],

    'baseline_start': '1990-01-01',
    'baseline_end': '2014-12-31',
    'hist_start': '1990-01-01',

    # (start, end, label, tag) -- same format as the interactive notebook's
    # "Future intervals" text box, one tuple per line.
    'future_intervals': [
        ('2026-01-01', '2050-12-31', 'Short', '2026-2050'),
        ('2051-01-01', '2075-12-31', 'Mid',   '2051-2075'),
        ('2076-01-01', '2100-12-31', 'Long',  '2076-2100'),
    ],

    'wet_months': [5, 6, 7, 8, 9, 10],
    'dry_months': [m for m in range(1, 13) if m not in [5, 6, 7, 8, 9, 10]],

    'temp_thresholds': [30.0],
    'precip_thresholds': [20.0, 25.0],
    'return_periods': [100],
    'gev_n_bootstrap': 1000,

    'nquantiles': 50,
    'wet_thresh': 0.1,

    'output_dir': '/content/drive/MyDrive/climate_output',   # <-- put this on Drive too
    'add_satellite_basemap': False,
}

## 3. Authenticate to Earth Engine via service account (non-interactive)

Reads the two secrets from step 3 of the setup above and hands them to
`run_pipeline` -- this is the new `gee_service_account` / `gee_key_data`
path added to `main.py` specifically so a scheduled run never hits the
interactive `ee.Authenticate()` browser prompt.

In [ ]:
from google.colab import userdata

params['gee_service_account'] = userdata.get('GEE_SA_EMAIL')
params['gee_key_data'] = userdata.get('GEE_SA_KEY_JSON')

print(f"Using service account: {params['gee_service_account']}")

## 4. Run -- no button, no confirmation. This is what a scheduled trigger executes.

In [ ]:
from main import run_pipeline

results = run_pipeline(params)
print("\n✅ Scheduled run complete.")